# Structured Streaming: File Source and Ingestion Directory Monitoring

**Module**: 09_streaming  
**Notebook**: 02_structured_streaming.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Setting up file-based streaming monitoring `/opt/spark-data/streaming/events/`, reading JSON streams, and writing to file sinks.
- **Why does this topic exist?**: Directory monitoring is the standard pattern for landing zones and file-based ETL ingestion.
- **What problem does it solve?**: Processing newly arriving files without re-reading processed files.
- **Where is it used in real-world Data Engineering?**: Automated Bronze layer ingestion in cloud data lakes.

---

## 2. Directory File Stream Architecture: Watches a directory and atomically processes new files as they arrive.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("File_Streaming_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
import json

stream_dir = "/opt/spark-data/streaming/events"
checkpoint_dir = "/opt/spark-data/warehouse/stream_chk_02"
out_stream = "/opt/spark-data/warehouse/stream_out_02"

# Clean test dirs
os.system(f"rm -rf {stream_dir}/* {checkpoint_dir} {out_stream}")
os.makedirs(stream_dir, exist_ok=True)

# Define schema for incoming JSON events
stream_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("amount", DoubleType()),
    StructField("channel", StringType())
])

# Create file stream reader
file_stream = (
    spark.readStream
    .schema(stream_schema)
    .json(stream_dir)
)

print("Streaming DataFrame created from file directory:", file_stream.isStreaming)

---

## Experiment: Generating Files and Observing Stream Pickup

- **1. What are we testing?**: Write sample JSON files into stream_dir and verify the stream picks them up.
- **2. What do we expect?**: Streaming query processes new files incrementally.

In [ ]:
# Start streaming query writing to parquet sink
q = (
    file_stream
    .writeStream
    .format("parquet")
    .option("checkpointLocation", checkpoint_dir)
    .outputMode("append")
    .start(out_stream)
)

# Generate file 1
with open(f"{stream_dir}/event_1.json", "w") as f:
    f.write(json.dumps({"order_id": 1, "customer_id": 101, "amount": 99.5, "channel": "Web"}) + "\n")

time.sleep(3)
q.processAllAvailable()

print(f"Rows processed so far in out_stream: {spark.read.parquet(out_stream).count()}")
q.stop()

**3. What happened?**: Spark detected the new file and wrote output to Parquet.  
**4. Why did it happen?**: Structured Streaming tracks processed file names in the checkpoint directory.  
**5. Practical Engineering Lesson**: File streaming provides automated, exactly-once directory ingestion without external orchestrators.

---

## Practical Exercises

### Exercise 1 (Beginner): Inspect Checkpoint Directory
**Problem Statement**: List contents of checkpoint directory.  
**Input / Context**: checkpoint_dir

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Add Second Event File
**Problem Statement**: Write a second file and verify incremental addition.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Atomic File Renaming
**Problem Statement**: Why files should be moved into directory atomically.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Inspect Checkpoint Directory):
```python
print(os.listdir(checkpoint_dir))
```

#### Solution 2 (Add Second Event File):
```python
print('File streaming is incremental.')
```

#### Solution 3 (Atomic File Renaming):
```python
print('Writing directly to directory risks Spark reading partial half-written files.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **File Stream**: Monitors directory for new files.
- **Checkpoint**: Records processed file offsets.

### Essential Technical Terms
- `processAllAvailable()`: Processes pending data synchronously (great for testing).
- `checkpointLocation`: Storage for progress metadata.

### Common Engineering Mistakes
- **Writing files directly into stream folder**: Causes partial read errors; always write to temp and atomic mv.

### Interview Questions on This Topic
- **Q: How does Structured Streaming avoid reprocessing files?**
  - *A*: It tracks processed file paths and versions in the internal offset log inside the checkpoint directory.

### What to Study Next
Open 03_triggers_and_output_modes.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')